# Truppenerkennung trainieren (Colab / Kaggle, GPU gratis)

Läuft komplett im Browser — **auch vom Handy**. Zellen der Reihe nach ausführen.

Vorher: **Laufzeit → Laufzeittyp ändern → GPU (T4)**. Ohne GPU dauert das Training Tage statt Stunden.

Was passiert:
1. Repo + Cutout-Datensatz holen
2. Synthetische Trainingsbilder erzeugen (Labels fallen dabei ab)
3. Val-Set aus **echten** Frames bauen — nach Episode getrennt
4. YOLO trainieren
5. Auf echten Bildern auswerten, **pro Klasse**
6. Nach ONNX exportieren

In [ ]:
!nvidia-smi || echo 'KEINE GPU -- Laufzeittyp auf GPU umstellen!'

In [ ]:
%cd /content
!pip -q install ultralytics pyyaml

# Eigener Code
![ -d Clash ] || git clone -b claude/clash-royale-bot-overview-ym909j https://github.com/gunthoradrian-arch/Clash.git

# Cutouts + echte, gelabelte Frames (MIT, ~1.3 GB)
![ -d crds ] || git clone --depth 1 https://github.com/wty-yy/Clash-Royale-Detection-Dataset.git crds

%cd /content/Clash
!git pull --ff-only || true

In [ ]:
# Was steckt drin? Klassenverteilung, Boxgrößen, Lücken.
!python tools/analyze_dataset.py --root /content/crds --md /content/report.md
print(open('/content/report.md').read()[:2500])

In [ ]:
N_SYNTH = 20000  # 20k reicht für yolov8s; auf T4 rund 30-40 min Erzeugung

!python tools/generate_dataset.py \
    --dataset-root /content/crds \
    --out /content/data/synth \
    -n $N_SYNTH --split train --preview 12 --seed 0

In [ ]:
# Sichtkontrolle VOR dem Training. Sehen die Szenen plausibel aus?
# Boxfarbe: blau = eigene Einheit, rot = gegnerische.
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob('/content/data/synth/preview/*.jpg'))[:4]:
    display(IPImage(p, width=430))

In [ ]:
# Val-Set aus echten Frames. --val-ratio 1.0 = alle echten Bilder validieren,
# nichts davon mittrainieren.
!python tools/prepare_real_val.py \
    --dataset-root /content/crds \
    --out /content/data/real \
    --val-ratio 1.0 --symlink

In [ ]:
# Ultralytics erwartet labels/ neben images/. Wir legen die 5-Feld-Variante
# dorthin; die 12-Feld-Labels bleiben unter labels_full/ für später.
import shutil, pathlib, yaml

root = pathlib.Path('/content/data/train_run')
if root.exists():
    shutil.rmtree(root)
(root / 'images').mkdir(parents=True)
(root / 'labels').mkdir(parents=True)

(root / 'images/train').symlink_to('/content/data/synth/images/train')
(root / 'labels/train').symlink_to('/content/data/synth/labels_yolo/train')
(root / 'images/val').symlink_to('/content/data/real/images/val')
(root / 'labels/val').symlink_to('/content/data/real/labels_yolo/val')

cfg = yaml.safe_load(open('/content/data/synth/data.yaml'))
cfg['path'] = str(root)
cfg['train'] = 'images/train'
cfg['val'] = 'images/val'
yaml.safe_dump(cfg, open(root / 'data.yaml', 'w'), sort_keys=False, allow_unicode=True)
print(f"{len(cfg['names'])} Klassen")
print(root / 'data.yaml')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='/content/data/train_run/data.yaml',
    epochs=60,
    imgsz=768,          # Arena ist 568x896 -- klein genug für Details, groß genug für Skelette
    batch=16,
    workers=2,
    patience=15,
    close_mosaic=10,    # letzte Epochen ohne Mosaic: stabilisiert kleine Objekte
    degrees=0.0,        # NICHT rotieren -- die Arena hat eine feste Orientierung
    fliplr=0.0,         # NICHT spiegeln -- links/rechts trägt Bedeutung (Lane)
    flipud=0.0,
    scale=0.25,
    hsv_h=0.012, hsv_s=0.5, hsv_v=0.35,
    project='/content/runs', name='cr_det', exist_ok=True,
)

In [ ]:
# Auswertung auf ECHTEN Bildern, pro Klasse.
# mAP50 allein mittelt die Probleme weg -- der Recall der kleinen Einheiten
# ist die Zahl, auf die es ankommt.
import numpy as np

m = YOLO('/content/runs/cr_det/weights/best.pt')
metrics = m.val(data='/content/data/train_run/data.yaml', imgsz=768, split='val')

print(f"mAP50    {metrics.box.map50:.3f}")
print(f"mAP50-95 {metrics.box.map:.3f}")

names = m.names
rows = []
for i, c in enumerate(metrics.box.ap_class_index):
    rows.append((names[int(c)], float(metrics.box.r[i]), float(metrics.box.p[i]),
                 float(metrics.box.ap50[i])))
rows.sort(key=lambda r: r[1])

print('\n--- 25 schwächste Klassen nach Recall ---')
print(f"{'Klasse':28s} {'Recall':>7s} {'Prec':>7s} {'AP50':>7s}")
for n, r, p, ap in rows[:25]:
    print(f'{n:28s} {r:7.3f} {p:7.3f} {ap:7.3f}')

print('\nGenau diese Klassen im Generator hochziehen oder Cutouts nachernten.')

In [ ]:
# Export für die Inferenz im Bot (in-process, kein HTTP).
m.export(format='onnx', imgsz=768, opset=12, simplify=True)
!ls -la /content/runs/cr_det/weights/

In [ ]:
# Gewichte sichern -- die Colab-Laufzeit wird sonst irgendwann weggeräumt.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/crbot
!cp /content/runs/cr_det/weights/best.pt   /content/drive/MyDrive/crbot/
!cp /content/runs/cr_det/weights/best.onnx /content/drive/MyDrive/crbot/
print('gesichert nach Drive/crbot/')